<a href="https://colab.research.google.com/github/tmulderrig/vida_yellowstone_finemapping/blob/main/zero_shot_vida_yellowstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Basic examples with PlantCaduceus

### Setup environment

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


## Install specific pytorch
We are installing a specific version of PyTorch (2.3.1) to resolve some errors when loading mamba-ssm, The most critical part is `--index-url https://download.pytorch.org/whl/cu121`. This command tells pip to NOT use the default repository. Instead, it downloads a special version of PyTorch that was pre-compiled specifically for systems with a CUDA 12.1 driver.

#### Why is this necessary?
Google Colab has its own system-level NVIDIA driver (e.g., CUDA 12.5). By installing a PyTorch build that is aware of a compatible CUDA version (12.1 works perfectly with a 12.5 driver), we ensure that all parts of the library, including low-level components like Triton, can find the correct drivers and compile code successfully during model inference. This fixes both the `libcuda.so` and the `ModuleNotFoundError` errors.

In [ ]:
!pip3 install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 23.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 77.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 60.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 34.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 77.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB ? eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 15.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB

In [ ]:
!pip3 install mamba-ssm==2.2.2 transformers==4.40.0 git+https://github.com/dridk/PyVCF3.git@master scipy==1.12.0 biopython xgboost==2.0.3 scikit-learn==1.4.0 matplotlib

  Cloning https://github.com/dridk/PyVCF3.git (to revision master) to /tmp/pip-req-build-1mmjxt8r
  Running command git clone --filter=blob:none --quiet https://github.com/dridk/PyVCF3.git /tmp/pip-req-build-1mmjxt8r
  Resolved https://github.com/dridk/PyVCF3.git to commit 1fb3789153d1d8e28e2cedf121399f276b5f312a
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 11.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.1/297.1 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12

🔄 You may need to restart the session after running the above setup cells.

Once restarted, you’re all set to start exploring the PlantCAD model!

## Testing if mamba-ssm is installed successfully

In [ ]:
# Test core dependencies
import torch
from mamba_ssm import Mamba
from transformers import AutoTokenizer, AutoModelForMaskedLM
import pandas as pd

device = 'cuda:0'

# Test PlantCAD model loading
tokenizer = AutoTokenizer.from_pretrained('kuleshov-group/PlantCaduceus_l32')
model = AutoModelForMaskedLM.from_pretrained('kuleshov-group/PlantCaduceus_l32', trust_remote_code=True)
model.to(device)
print("✅ Installation successful!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/419 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

modeling_caduceus.py: 0.00B [00:00, ?B/s]

modeling_rcps.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/kuleshov-group/PlantCaduceus_l32:
- modeling_rcps.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/kuleshov-group/PlantCaduceus_l32:
- modeling_caduceus.py
- modeling_rcps.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/902M [00:00<?, ?B/s]

✅ Installation successful!


## Run some examples on zero-shot mutation effect prediction

In [ ]:
# clone PlantCAD repo
!git clone https://github.com/kuleshov-group/PlantCaduceus.git

Cloning into 'PlantCaduceus'...
remote: Enumerating objects: 667, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 667 (delta 111), reused 113 (delta 69), pack-reused 497 (from 1)
Receiving objects: 100% (667/667), 37.12 MiB | 16.59 MiB/s, done.
Resolving deltas: 100% (309/309), done.


In [ ]:
!wget https://urgi.versailles.inrae.fr/download/iwgsc/IWGSC_RefSeq_Assemblies/v1.0/iwgsc_refseqv1.0_chr2D.fsa.zip
!unzip iwgsc_refseqv1.0_chr2D.fsa.zip

--2026-02-19 15:53:55--  https://urgi.versailles.inrae.fr/download/iwgsc/IWGSC_RefSeq_Assemblies/v1.0/iwgsc_refseqv1.0_chr2D.fsa.zip
Resolving urgi.versailles.inrae.fr (urgi.versailles.inrae.fr)... 138.102.159.161
Connecting to urgi.versailles.inrae.fr (urgi.versailles.inrae.fr)|138.102.159.161|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 193787592 (185M) [application/zip]
Saving to: ‘iwgsc_refseqv1.0_chr2D.fsa.zip.3’

iwgsc_refseqv1.0_ch 100%[===================>] 184.81M  74.1MB/s    in 2.5s    

2026-02-19 15:53:58 (74.1 MB/s) - ‘iwgsc_refseqv1.0_chr2D.fsa.zip.3’ saved [193787592/193787592]

Archive:  iwgsc_refseqv1.0_chr2D.fsa.zip
replace iwgsc_refseqv1.0_chr2D.fsa? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [ ]:
# Upload VCF
from google.colab import files
uploaded = files.upload()
uploaded_vcf = list(uploaded.keys())[0]


# Decompress if needed
!gunzip -c "{uploaded_vcf}" > input.vcf || cp "{uploaded_vcf}" input.vcf

Saving filtered_vida_yellowstone_snps.vcf to filtered_vida_yellowstone_snps.vcf


In [ ]:
# Download wheat chromosome 4A reference sequence
!wget https://urgi.versailles.inrae.fr/download/iwgsc/IWGSC_RefSeq_Assemblies/v1.0/iwgsc_refseqv1.0_chr4A.fsa.zip

!unzip -o iwgsc_refseqv1.0_chr4A.fsa.zip


--2026-04-14 20:01:19--  https://urgi.versailles.inrae.fr/download/iwgsc/IWGSC_RefSeq_Assemblies/v1.0/iwgsc_refseqv1.0_chr4A.fsa.zip
Resolving urgi.versailles.inrae.fr (urgi.versailles.inrae.fr)... 138.102.159.161
Connecting to urgi.versailles.inrae.fr (urgi.versailles.inrae.fr)|138.102.159.161|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 221930949 (212M) [application/zip]
Saving to: ‘iwgsc_refseqv1.0_chr4A.fsa.zip.1’

iwgsc_refseqv1.0_ch 100%[===================>] 211.65M  5.61MB/s    in 18s     

2026-04-14 20:01:38 (11.6 MB/s) - ‘iwgsc_refseqv1.0_chr4A.fsa.zip.1’ saved [221930949/221930949]

Archive:  iwgsc_refseqv1.0_chr4A.fsa.zip
  inflating: iwgsc_refseqv1.0_chr4A.fsa  


In [ ]:
#Run PlantCaduceus
!python PlantCaduceus/src/zero_shot_score.py \
    -input-vcf input.vcf \
    -input-fasta iwgsc_refseqv1.0_chr4A.fsa \
    -output zero_shot_4A.vcf \
    -model 'kuleshov-group/PlantCaduceus_l32' \
    -device 'cuda:0'

2026-04-14 20:41:18 - WARNING - PlantCaduceus (PlantCAD v1) model detected. Forcing contextSize=512 (was 2048).
2026-04-14 20:41:18 - INFO - Reading input data from input.vcf
2026-04-14 20:41:18 - INFO - Loading reference genome from iwgsc_refseqv1.0_chr4A.fsa
2026-04-14 20:41:23 - INFO - Loading model and tokenizer from kuleshov-group/PlantCaduceus_l32
2026-04-14 20:41:23 - INFO - Using float16 as the GPU supports sm_60 or higher.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn

In [ ]:
files.download('zero_shot_4A.vcf')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!grep -v '#' zero_shot_4A.vcf | awk  -v OFS='\t' '{print $1,$2,$4,$5,$8}' | head -n 20

chr4A	629478003	T	C	AC=146;AF=0.23;AN=636;BaseQRankSum=-0.358;ClippingRankSum=-0.736;DP=1924;FS=0;GQ_MEAN=16.52;GQ_STDDEV=11.88;InbreedingCoeff=0.751;MLEAC=142;MLEAF=0.223;MQ=60.0;MQ0=0;MQRankSum=0.747;NCC=19;QD=26.92;ReadPosRankSum=0.736;plantCAD_zero_shot=4.5546875
chr4A	629510461	G	A	AC=11;AF=0.024;AN=460;BaseQRankSum=-1.026;ClippingRankSum=-1.026;DP=622;FS=0;GQ_MEAN=7.99;GQ_STDDEV=5.27;InbreedingCoeff=0.0838;MLEAC=9;MLEAF=0.02;MQ=60.0;MQ0=0;MQRankSum=-1.026;NCC=107;QD=20.34;ReadPosRankSum=0;plantCAD_zero_shot=-0.61120605
chr4A	629510501	A	T	AC=12;AF=0.024;AN=498;DP=810;FS=0;GQ_MEAN=9.41;GQ_STDDEV=6.94;InbreedingCoeff=0.1348;MLEAC=9;MLEAF=0.018;MQ=60.0;MQ0=0;NCC=88;QD=23.47;plantCAD_zero_shot=-0.47644043
chr4A	629598526	T	C	AC=130;AF=0.313;AN=416;BaseQRankSum=0.736;ClippingRankSum=1.29;DP=722;FS=9.633;GQ_MEAN=12.89;GQ_STDDEV=17.75;InbreedingCoeff=0.4635;MLEAC=138;MLEAF=0.332;MQ=60.0;MQ0=0;MQRankSum=0.736;NCC=129;QD=23.57;ReadPosRankSum=0.742;plantCAD_zero_shot=0.060546864
chr4A	6298